In [59]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

print("Installing required packages...")

!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-groq \
langchain-core \
faiss-cpu \
pypdf \
sentence-transformers \
streamlit \
langsmith

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Installing required packages...


In [60]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

LANGCHAIN_API_KEY = user_secrets.get_secret("LANGCHAIN_API_KEY")
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

print("LangSmith Key Loaded:", LANGCHAIN_API_KEY[:10], "...")
print("Groq Key Loaded:", GROQ_API_KEY[:10], "...")

LangSmith Key Loaded: lsv2_pt_0e ...
Groq Key Loaded: gsk_k9WBYM ...


In [61]:
import os
from kaggle_secrets import UserSecretsClient
from cryptography.fernet import Fernet

user_secrets = UserSecretsClient()

os.environ["LANGCHAIN_API_KEY"] = user_secrets.get_secret("LANGCHAIN_API_KEY")
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "zyro-rag-challenge"

SUBMISSION_SECRET = b"6Q_EBPtBG-60URcrF6jxNTJSRjy-CtZbJlvp_xf0c_M="
fernet = Fernet(SUBMISSION_SECRET)

print("Environment configured successfully!")

Environment configured successfully!


In [62]:
import os
from langchain_community.document_loaders import PyPDFLoader

# Folder containing all HR policy PDFs
pdf_folder = "/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus"

documents = []

# Load every PDF
for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder, file))
        documents.extend(loader.load())

print(f"Loaded {len(documents)} pages from all PDFs.")

Loaded 39 pages from all PDFs.


In [63]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

# Split the documents into chunks
chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks.")

Created 168 chunks.


In [64]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model initialized successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model initialized successfully!


In [65]:
from langchain_community.vectorstores import FAISS

# Build the FAISS vector database
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Create an MMR retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10
    }
)

print("✅ Vector Store created successfully!")
print("Total vectors:", vectorstore.index.ntotal)

✅ Vector Store created successfully!
Total vectors: 168


In [66]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=512
)

print("✅ Groq LLM initialized successfully!")

✅ Groq LLM initialized successfully!


In [67]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Prompt Template
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are the official HR Assistant for Zyro Dynamics.

Use ONLY the provided context to answer the user's question.

Rules:
1. Do not make up information.
2. If the answer is clearly present in the context, answer briefly and professionally.
3. If the context does not contain enough information, reply exactly:
"Based on the available Zyro Dynamics HR policy documents, I couldn't find information that answers this question."

Context:
{context}

Question:
{question}

Answer:
""")

# Convert retrieved documents into one string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Main RAG function
def rag_chain(question: str):
    docs = retriever.invoke(question)

    context = format_docs(docs)

    response = (
        RAG_PROMPT
        | llm
        | StrOutputParser()
    ).invoke({
        "context": context,
        "question": question
    })

    return {
        "answer": response,
        "sources": docs
    }

print("✅ RAG Chain created successfully!")

✅ RAG Chain created successfully!


In [68]:
question = "How many earned leaves are employees entitled to?"

result = rag_chain(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(result["answer"])

print("\nSOURCE DOCUMENTS:")
for i, doc in enumerate(result["sources"], 1):
    print(f"\nSource {i}:")
    print(doc.metadata)

QUESTION:
How many earned leaves are employees entitled to?

ANSWER:
15 days of Earned Leave upon completion of one year of continuous service.

SOURCE DOCUMENTS:

Source 1:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-20T10:58:03+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-20T10:58:03+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/02_Leave_Policy.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}

Source 2:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-20T10:58:03+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-20T10:58:03+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus

In [69]:
REFUSAL_MESSAGE = (
    "Sorry, I can only answer questions related to Zyro Dynamics HR policies."
)

def ask_bot(question: str):
    docs = retriever.invoke(question)

    # If no relevant documents found
    if len(docs) == 0:
        return {
            "answer": REFUSAL_MESSAGE,
            "sources": []
        }

    context = format_docs(docs)

    # Simple relevance check
    if len(context.strip()) < 50:
        return {
            "answer": REFUSAL_MESSAGE,
            "sources": []
        }

    response = (
        RAG_PROMPT
        | llm
        | StrOutputParser()
    ).invoke({
        "context": context,
        "question": question
    })

    return {
        "answer": response,
        "sources": list(set(
            os.path.basename(doc.metadata["source"])
            for doc in docs
        ))
    }

print("✅ Guardrails added successfully!")

✅ Guardrails added successfully!


In [70]:
test_questions = [
    "How many earned leaves are employees entitled to?",
    "Can I work from home permanently?",
    "Who won the FIFA World Cup?"
]

for i, q in enumerate(test_questions, 1):
    print("=" * 70)
    print(f"Question {i}: {q}")

    result = ask_bot(q)

    print("\nAnswer:")
    print(result["answer"])

Question 1: How many earned leaves are employees entitled to?

Answer:
15 days of Earned Leave upon completion of one year of continuous service.
Question 2: Can I work from home permanently?

Answer:
Based on the available Zyro Dynamics HR policy documents, I couldn't find information that answers this question.
Question 3: Who won the FIFA World Cup?

Answer:
Based on the available Zyro Dynamics HR policy documents, I couldn't find information that answers this question.


In [71]:
print("""
HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!
""")


HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!



In [72]:
app_code = r'''
import streamlit as st
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ---------------------------
# Streamlit Page
# ---------------------------

st.set_page_config(
    page_title="Zyro Dynamics HR Assistant",
    page_icon="🤖"
)

st.title("🤖 Zyro Dynamics HR Assistant")
st.write("Ask any question about Zyro Dynamics HR policies.")

# ---------------------------
# Load API Key
# ---------------------------

groq_key = st.secrets["GROQ_API_KEY"]

llm = ChatGroq(
    groq_api_key=groq_key,
    model="llama-3.3-70b-versatile",
    temperature=0
)

# ---------------------------
# Load Documents
# ---------------------------

@st.cache_resource
def load_vectorstore():

    pdf_folder = "docs"

    documents = []

    for file in os.listdir(pdf_folder):
        if file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(pdf_folder, file))
            documents.extend(loader.load())

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )

    chunks = splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )

    return vectorstore


vectorstore = load_vectorstore()

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,
        "fetch_k":10
    }
)

# ---------------------------
# Prompt
# ---------------------------

prompt = ChatPromptTemplate.from_template("""
You are the official HR Assistant for Zyro Dynamics.

Use ONLY the provided context to answer the user's question.

Rules:
1. Never make up information.
2. Answer only from the provided context.
3. If the answer is not available in the context, reply exactly:

"Based on the available Zyro Dynamics HR policy documents, I couldn't find information that answers this question."

Context:
{context}

Question:
{question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def ask_bot(question):

    docs = retriever.invoke(question)

    context = format_docs(docs)

    response = (
        prompt
        | llm
        | StrOutputParser()
    ).invoke({
        "context": context,
        "question": question
    })

    sources = sorted(
        list(
            set(
                os.path.basename(doc.metadata["source"])
                for doc in docs
            )
        )
    )

    return response, sources

# ---------------------------
# Chat Interface
# ---------------------------

question = st.text_input(
    "Ask your HR question:",
    placeholder="Example: How many earned leaves are employees entitled to?"
)

if st.button("Get Answer"):

    if question.strip() == "":
        st.warning("Please enter a question.")

    else:
        with st.spinner("Searching HR policies..."):

            answer, sources = ask_bot(question)

        st.subheader("Answer")
        st.write(answer)

        st.subheader("Source Documents")

        for src in sources:
            st.write(f"• {src}")

'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py created successfully!")

✅ app.py created successfully!


In [73]:
_Q = [
    ("Q01", "gAAAAABqE-m-EnBhR94RLAsyCD5YUOimCgpyxnGmrg3N29dvcCChh_LbQzGhacqtB6Rg9ySTN-aO4eS5nnSSqgvslxWg3T2XNxvKRw9BoZOGB8sSrPpeXOqPKhdprAkvepa0Ef13rK84Lx_QKNPq5AMeO2zweDFo-UGpOZ1yFV_k0NbpkP0MshR9BpjCI4QpkDSx9QH95aeCK8sqSIkcM8wOFRs1hRD_tV-Jg4XmeHLm4jW6wpCWQRBF-XWIHTwCE3Tod-Zfj-nIFpPe3sNmXFDNY_L5g8aAiw=="),
    ("Q02", "gAAAAABqE-m-iGIUkxaPu-TWqkoQqfrY1QvCn-VC445z8EzeRjBVVSjcBgTYC-OS2QVoM37Oh8tFkJdLJcdivCIg9-jTJ72Vy24BQwagKYrIJlkNBr9yectRVtDZ_X24PWpsbIdMcelH1a6VBz9XXmJ19-0HvqFT0kTeEQEyjzKL2BmtoSHOquqe74xGFhpWD-fI1Cshfxk9EXwgA4poqi7JJ3ovja5pVM18uwfNAmcNacnQRtFTAm6x1JmXKSYVeBSbgpOv1zjEEC-0XfVhF0Wtwli0hRZHhA=="),
    ("Q03", "gAAAAABqE-m-qhjI3OCH68smnD4afuA_GmeOO8rI6R79iaPeodfwbt4NTlWhlbSfgr8BP9ZNAi5yczk65fgsIgbRXQ9AkAVDE2NOD11Aqt6U_NqURkjBQpzn3gzTQNj2qNwtkhx71-l8uYIfZLu8Z-Nv4aAkEaFTKCDp4DWgCaFJbe90TCA2fGUVnDiaI1_0ID87AHR-yYRwTaKYiWI7PiCQWFVm22NGx3cwX_uvMouAEXLX2sw_o3s="),
    ("Q04", "gAAAAABqE-m-qVKLekYizIYVBejJAmZYhT0zftdQzC0nbFt6BAJM52tiRsM0y5pcEfTl7y2bKwjFBSBwj3ik1P1yPTz6mP2h1xHEWoeJxPHdvujlZXJv8ObZO0PbHSPMk6xtnEmEqPAfPLzxjOzu63P3K_0eFdpgR48fUbcQwZt7yZkGzOPqYuUDAE7CBmvgvwRfwymkEzTD8ESt0vYvZdmoYjV7sbScmhoxYbWmjMatFvOzha6D1YA="),
    ("Q05", "gAAAAABqE-m-KRbrY2MpEseeszU46iQWHzbzwOO5-t10vHJrdQOKeaVwPxyp9kiBDCS1Fa5MJyQoTOp2pdEtw9LtUbCEJ_56caOBjtBgngLz4kvcodhVECBLBuD6vsCaQlopu0SardsvA3slA379M8nrcyuuea3dJ97FPlOdQs2b70BRPyOkyNH0nKGqBwQzBlAW7B-ucZwf9dDPPAw-xUTfR3ekIqXReQ=="),
    ("Q06", "gAAAAABqE-m-EYfgWBpxkb_5hGOvvBsAdBu5367Nd5d4uT_6EEAaTeCidG99u5XJ5vcZatZpoj5RjmfrY5O1XNObuApuq_ZFah_StEcLHB31Ow6WRrZpikDGUFJkC-ZfY0TggJzDFvdtwQsIttqNW5js0LMS-74V-AUx0UCi4bABm1vOMGBKP2qGyGTfyh2wfETTw4nNhbac"),
    ("Q07", "gAAAAABqE-m-cZLyG6To-HyWWdEYu42VgbV9c_SCWXt4qJE02YrOFvfMntuBTf-CVXt3MhJWFzrukGMR0-Brla1QMVbefRelzpJqkY2TsIQ3Tcc5MZ0BH6ornHjZAnOd9Iozf1f755EC8hBase1XtbhThrKgYJRKWPxaxKd-nkLK3XuabtmEF8r0bZtTyKVjYNBUWPT--lKJb-pXvw3p3zJ0z6utBLWicmBhgdJvGMoOQCsCLrxi6jrtHZzka7Me7Vm6UUhwSkdz"),
    ("Q08", "gAAAAABqE-m-sxXijCcjguEWTh7qgKt7BX4cbUfFdUwAz6VqSoU4fTnYXUhf-dVQdCKa1lhgc7ZZatU5Pu9iuQHG-ApZCOw2yR-PkZnuY9L7uR02CCJoWYhFQelqYEWYA5uONridoCzD8kh2yqwUSVInEFfBuB2cYgyPobRnP_yRvtaFtLakrMy0fsCZH_zfyrOMVkdF5GoHdPu67XzoEj806x4aS8DJ4ysYFuwNb9zkhhceq_CsU08="),
    ("Q09", "gAAAAABqE-m-nDGYgCF3fSWs2tM39pdnsBua61Ht1ruTZ_NOUmju6AxbGU6WB8HzLEHKQkkCnxc4ka2DohiUSLwVDrWG2ZnGggyt7OnI6D43ovjDBsMhW2jQPaz9zaHua25abfEqF4V1ZioQrdL7lz3D0qzDsjXl4Kw5RY2g3kaDakb62Cb6Dt8badoS-t4Bd_fEAp49t09FH_qwLp_ZTotiFsKFy6QADA=="),
    ("Q10", "gAAAAABqE-m-PwoVsLjWO4nbO8W_65P-UNNF7SjdNZL4sRN-G72eHygPuGyggXwVG8G7HJ2ZmrtCYuNg-rtWH_iuyexPQLVG0EqKr0ZQswJox4iauvFf014qlqr5vC_TtdwHGcMiZsyWZpJauDTffKDm_QJHrGElPUUunCFgX8356s1yMocleGXUBfcZ8B73A5LIALAXRIBpKyt707qYlLhwOG1vhsdR74q21NS0-n0skLZIy7z0pLM="),
    ("Q11", "gAAAAABqE-m-1BAGkhsZEDnkbSwAAwusmnMKdn2gvIM5tltaZ1W-eoKtvbPNu8rkAlOOiOW-9_NobJqDFKDO3J7zCPwWuEdGxwgYpX5sxh2Rg4ngR5R5WDnQsQTPIRHXJkkaN1ufNhvbQ-XOn2Z1QPci8118ByVpkAR5kZTUXOFIZ1IgHP2hbvO4E81GB9CTs9HiZvHAsAnS"),
    ("Q12", "gAAAAABqE-m-NrwI-KspXny9JlQqBEW_eB9jE6bGmnin6IX6SdcB9ol1gR7CmzczDKE6A7XHDOJW20tVHAlGFw-q-J6cWrTajK_mJTv00aHllSozrKiThojuxxnSjhgOhgtNKU5mh7zoz2d2uLo7p-Kl32m4IU6PRsm0kZceID-ZH5ZRw7w4h1qSZOufZO2HvKkR9LtfCQXk"),    
    ("Q13", "gAAAAABqE-m-Xr56G8qaFfk3BIVQeDzP5mpahd7wZQ5vGR11AN_sxU1ZzjoPfbSdLmrrhFHEI8S8KhXfjOWZQoMJToWSsnhjZQdrRj0wujH38p2VOZLqqZYSmOflVEQm29z9pAXx_iltLWZLNGf8QsMtZWuo-3SsWt6R2mGvOMBTDj5hCzaq842_r1eupRQJJ1dnTSmNPskW"),
    ("Q14", "gAAAAABqE-m--oxJAL26EQ6bMS5vmgI0pWMWjgbG49qNZu8K_pIiDrp3ro1YFlVvBXOOJ6bSpI7lxz-OXmNrVFkSfJlVf4PchVKfWdddKVT85AMxUHo3PYD15IGV476RznHCiD59twp7x_E6HOF7AFUGiWcsO9Ph63Tfcvh3aJzF7Hk_NPEHcIaaEU9ki2eccYXehJJ3tkmr"),
    ("Q15", "gAAAAABqE-m-3JNAfb2dmCF-2XlNe-F1AaeXybgSJ4DwHtn9o52TEryPYgu-6m70Ivn7izeLy4h44AVbHL_3cv-MWfAwFYp7ct3lvF7dL1QbmhntyeY4c7l0CVPsc-mv8WuY04tpB2XPtHE_0ytl9tQlqAGonC2esnpMbSzgvZPdSw9eHnm5k2Jkh0FbgjLKNWxjdX3Uv2aYDiqOeLMQKZsMMteZzJcwHQ==")]
     
eval_questions = [
    {"question_id": qid, "question": fernet.decrypt(enc.encode()).decode()}
    for qid, enc in _Q
]

print(f"{len(eval_questions)} evaluation questions loaded.")

15 evaluation questions loaded.


In [ ]:
import re
import time
import csv

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("=" * 50)
print("Submission Generator")
print("=" * 50)

streamlit_link = input("Streamlit App URL:  ").strip()
langsmith_link = input("LangSmith Trace URL: https://smith.langchain.com/public/11e252b4-f548-4d31-9dd2-07d2b1a5211a/r/019f09e7-1a8c-71c2-aac0-0f164f5811eb").strip()

link_errors = []

if not STREAMLIT_PATTERN.match(streamlit_link):
    link_errors.append("Invalid Streamlit URL.")

if not LANGSMITH_PATTERN.match(langsmith_link):
    link_errors.append("Invalid LangSmith URL.")

if link_errors:
    print("\n".join(link_errors))
    raise ValueError("Please correct the links and re-run the cell.")

print(f"\nGenerating responses for {len(eval_questions)} questions...\n")

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "question_enc": fernet.encrypt(question.encode()).decode(),
        "answer_enc": fernet.encrypt(answer.encode()).decode(),
        "streamlit_link": streamlit_link,
        "langsmith_link": langsmith_link,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

csv_path = "submission.csv"

fieldnames = [
    "question_id",
    "question_enc",
    "answer_enc",
    "streamlit_link",
    "langsmith_link"
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nsubmission.csv generated successfully.")

In [75]:
requirements = """streamlit
langchain
langchain-community
langchain-text-splitters
langchain-huggingface
langchain-groq
langchain-core
langsmith
faiss-cpu
pypdf
sentence-transformers
transformers
torch
huggingface_hub
groq
python-dotenv
tiktoken
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt created successfully!")

✅ requirements.txt created successfully!


In [76]:
import os
print(os.path.exists("requirements.txt"))

True
